# FX spot and forwards

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Prerequisites:** `02_pricing/pricing_across_asset_classes.ipynb`. **`fx_spot`** and **`fx_forward`** use typed **`FxMatrix`** quotes that round-trip through the market snapshot.


## Concept

- **Spot:** immediate exchange of notionals at the dealt rate.
- **Forward:** covered interest parity links FX forward to domestic/foreign discount curves.
- **Multi-currency exposure:** notionals are tagged with **`currency`**; PV lands in the instrument’s settlement currency.


In [1]:
import json

import sys
sys.path.insert(0, "../..")

from _shared import DEMO_AS_OF, print_metrics, usd_ois_curve
from finstack_quant.valuations import ValuationResult
from finstack_quant.valuations.instruments import price_instrument_with_metrics, validate_instrument_json
from finstack_quant.core.currency import Currency
from finstack_quant.core.market_data import DiscountCurve, FxMatrix, MarketContext

print("Imports OK (FX).")


Imports OK (FX).


## Instrument JSON


In [2]:
AS_OF = DEMO_AS_OF
AS_OF_STR = AS_OF.isoformat()

fx_spot = json.dumps(
    {
        "type": "fx_spot",
        "spec": {
            "id": "EURUSD-SPOT",
            "base_currency": "EUR",
            "quote_currency": "USD",
            "notional": {"amount": "1000000", "currency": "EUR"},
            "spot_rate": 1.10,
            "settlement_lag_days": 2,
            "bdc": "modified_following",
            "attributes": {},
            "pricing_overrides": {},
        },
    }
)

fx_fwd = json.dumps(
    {
        "type": "fx_forward",
        "spec": {
            "id": "EURUSD-FWD-6M",
            "base_currency": "EUR",
            "quote_currency": "USD",
            "notional": {"amount": "1000000", "currency": "EUR"},
            "maturity": "2025-07-15",
            "contract_rate": 1.12,
            "domestic_discount_curve_id": "USD-OIS",
            "foreign_discount_curve_id": "EUR-OIS",
            "attributes": {"tags": ["fx"], "meta": {"pair": "EURUSD"}},
            "pricing_overrides": {},
        },
    }
)

for label, js in (("fx_spot", fx_spot), ("fx_forward", fx_fwd)):
    validate_instrument_json(js)
    print(label, "JSON OK")


fx_spot JSON OK
fx_forward JSON OK


## MarketContext


In [3]:
ois_usd = usd_ois_curve(AS_OF)
ois_eur = DiscountCurve(
    "EUR-OIS",
    AS_OF,
    [(0.0, 1.0), (0.5, 0.988), (1.0, 0.975), (3.0, 0.92), (5.0, 0.85), (10.0, 0.70)],
    day_count="act_365f",
)
fx = FxMatrix()
fx.set_quote(Currency("EUR"), Currency("USD"), 1.10)
mc = MarketContext().insert(ois_usd).insert(ois_eur)
mc.insert_fx(fx)
market_json = mc.to_json()
restored_fx = MarketContext.from_json(market_json).fx
print("EUR/USD quote:", restored_fx.rate(Currency("EUR"), Currency("USD"), AS_OF).rate)

EUR/USD quote: 1.1


## Pricing


In [4]:
for label, js in (("fx_spot", fx_spot), ("fx_forward", fx_fwd)):
    out = price_instrument_with_metrics(js, market_json, AS_OF_STR, model="discounting")
    print(label, ValuationResult.from_json(out))


fx_spot ValuationResult(id="EURUSD-SPOT", price=1100000.0000, currency=USD, metrics=0)
fx_forward ValuationResult(id="EURUSD-FWD-6M", price=-16425.0531, currency=USD, metrics=0)


## Metrics


In [5]:
metrics = ["fx01", "fx_delta", "spot_rate"]
for label, js in (("fx_spot", fx_spot), ("fx_forward", fx_fwd)):
    out = price_instrument_with_metrics(
        js, market_json, AS_OF_STR, model="discounting", metrics=metrics
    )
    print("—", label)
    print_metrics(ValuationResult.from_json(out), metrics)


— fx_spot
  fx01: 0.0
  fx_delta: 11000.0
  spot_rate: 1.1
— fx_forward
  fx01: 10869.130397917765


## Analytic: Quanto (quanto_option_price)

`quanto_option_price` is a closed-form quanto (equity/FX-adjusted) pricer. It is independent of the full instrument JSON pipeline but useful for quick cross-currency option checks.


In [6]:
from finstack_quant.valuations import quanto_option_price

q_price = quanto_option_price(
    spot=100.0,
    strike=100.0,
    t=1.0,
    rate_domestic=0.05,
    rate_foreign=0.03,
    div_yield=0.01,
    vol_asset=0.20,
    vol_fx=0.10,
    correlation=0.30,
    is_call=True,
)
print("Quanto ATM call (domestic units):", round(q_price, 6))

Quanto ATM call (domestic units): 8.319663


## Takeaways

- **Typed `FxMatrix` quotes** survive `MarketContext.to_json()` / `from_json()` and feed FX instruments directly.
- **Two discount curves** anchor forward FX via interest parity.
- FX metrics (`fx01`, `fx_delta`, …) are registry-dependent.
